# Phase 4a: Proposed Method - Architecture & Components

## GNN-Guided RL Retriever for GraphRAG (GRL-RAG)

### Two approaches (compare experimentally):

**Approach A: RL Path Traversal** (original)
- RL agent walks on local subgraph, guided by GNN scores
- Collects triples along the way

**Approach B: RL Path Selection** (alternative, potentially stronger)
- GNN-RAG generates ALL candidate paths (shortest paths to top candidates)
- RL agent SELECTS which paths to include in context (budget = max paths)
- Avoids "RL on small graph" problem, builds directly on GNN-RAG output

### Also includes:
- **GreedyGNN baseline**: simply follow GNN scores greedily (no RL)
  - If GreedyGNN ~= GRL-RAG, RL is not needed
- **Relation encoding**: actions include relation type (not just destination entity)

### Data flow:
```
P2 output:
  gnn_features_train.pkl  -> P4b: RL training (NO leakage)
  gnn_features_test.pkl   -> P4b: evaluation only
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/ThS/Research/colab/graph_rag')
from common import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import pickle
import json
from tqdm import tqdm
from collections import defaultdict, deque
from sentence_transformers import SentenceTransformer

## 1. GNN Heuristic Module (same as before, + answers field)

In [ ]:
class GNNHeuristic:
    """Wrapper for GNN features from Phase 2.
    
    Loads gnn_features_{split}.pkl with per-question:
    node_scores, top_candidates, q_entities, answers, subgraph_tuples
    """
    
    def __init__(self, features_path=None):
        self.features = {}
        if features_path and os.path.exists(features_path):
            with open(features_path, 'rb') as f:
                self.features = pickle.load(f)
            print(f'Loaded GNN features for {len(self.features)} questions')
        else:
            print('GNNHeuristic: No features loaded')
    
    def get(self, qid):
        return self.features.get(qid)
    
    def get_node_score(self, qid, entity):
        qdata = self.features.get(qid, {})
        return qdata.get('node_scores', {}).get(entity, 0.0)
    
    def get_local_graph(self, qid):
        qdata = self.features.get(qid)
        if qdata is None:
            return nx.Graph()
        G = nx.Graph()
        for triplet in qdata['subgraph_tuples']:
            h, r, t = triplet
            if isinstance(h, dict): h = h['text']
            if isinstance(r, dict): r = r['text']
            if isinstance(t, dict): t = t['text']
            # Convert to str (entity/relation co the la int ID)
            G.add_edge(str(h), str(t), relation=str(r).strip())
        return G
    
    def get_seeds(self, qid):
        qdata = self.features.get(qid, {})
        result = []
        for e in qdata.get('q_entities', []):
            result.append(e.get('text', str(e)) if isinstance(e, dict) else str(e))
        return result
    
    def get_answers(self, qid):
        qdata = self.features.get(qid, {})
        result = []
        for a in qdata.get('answers', []):
            result.append(a.get('text', a.get('kb_id', str(a))) if isinstance(a, dict) else str(a))
        return result

## 2. Entity & Relation Embedder

Encode BOTH entity names AND relation types with SBERT.

In [ ]:
class Embedder:
    """SBERT encoder for entities AND relations. Cached."""
    
    def __init__(self, model_name='all-MiniLM-L6-v2', cache_path=None):
        self.model = SentenceTransformer(model_name)
        self.emb_dim = self.model.get_sentence_embedding_dimension()
        self.cache = {}
        if cache_path and os.path.exists(cache_path):
            with open(cache_path, 'rb') as f:
                self.cache = pickle.load(f)
            print(f'Loaded {len(self.cache)} cached embeddings')
    
    def encode(self, text):
        if text is None:
            return torch.zeros(self.emb_dim)
        if text not in self.cache:
            emb = self.model.encode(text, convert_to_tensor=True)
            self.cache[text] = emb.cpu()
        return self.cache[text]
    
    def encode_batch(self, texts):
        uncached = [t for t in texts if t not in self.cache and t is not None]
        if uncached:
            embs = self.model.encode(uncached, convert_to_tensor=True, show_progress_bar=False)
            for name, emb in zip(uncached, embs):
                self.cache[name] = emb.cpu()
        return torch.stack([self.cache.get(t, torch.zeros(self.emb_dim)) if t else torch.zeros(self.emb_dim) for t in texts])
    
    def save_cache(self, path):
        with open(path, 'wb') as f:
            pickle.dump(self.cache, f)
        print(f'Saved {len(self.cache)} embeddings to {path}')


EMB_CACHE = os.path.join(CHECKPOINT_DIR, 'embeddings_cache.pkl')
embedder = Embedder(cache_path=EMB_CACHE)
EMB_DIM = embedder.emb_dim
print(f'Embedding dim: {EMB_DIM}')

## 3. Baseline: Greedy GNN Retriever

**Critical baseline**: simply follow GNN scores greedily without RL.
If this performs ~= GRL-RAG, then RL adds no value.

In [ ]:
class GreedyGNNRetriever:
    """Baseline: greedily follow highest GNN-scored neighbors.
    No RL, no learned policy. Pure GNN heuristic.
    """
    
    def __init__(self, gnn_heuristic, max_steps=10, budget=50):
        self.gnn = gnn_heuristic
        self.max_steps = max_steps
        self.budget = budget
    
    def retrieve(self, qid):
        qdata = self.gnn.get(qid)
        if qdata is None:
            return {'triples': [], 'cost': 0, 'context': ''}
        
        G = self.gnn.get_local_graph(qid)
        seeds = self.gnn.get_seeds(qid)
        
        visited = set(seeds)
        all_triples = []
        
        # BFS-like: expand from seeds, always pick highest GNN-scored neighbor
        frontier = list(seeds)
        steps = 0
        
        while frontier and steps < self.max_steps and len(visited) < self.budget:
            next_frontier = []
            # Collect all unvisited neighbors with GNN scores
            candidates = []
            for node in frontier:
                if node not in G:
                    continue
                for neighbor in G.neighbors(node):
                    if neighbor in visited:
                        continue
                    score = self.gnn.get_node_score(qid, neighbor)
                    rel = G[node][neighbor].get('relation', 'related_to')
                    candidates.append((node, rel, neighbor, score))
            
            if not candidates:
                break
            
            # Pick top by GNN score
            candidates.sort(key=lambda x: -x[3])
            for src, rel, dst, score in candidates:
                if len(visited) >= self.budget:
                    break
                if dst not in visited:
                    visited.add(dst)
                    all_triples.append((src, rel, dst))
                    next_frontier.append(dst)
            
            frontier = next_frontier
            steps += 1
        
        context = path_to_string(all_triples)
        return {'triples': all_triples, 'cost': len(visited), 'context': context}


def path_to_string(triples):
    if not triples:
        return ''
    parts = []
    for h, r, t in triples:
        if not parts:
            parts.append(str(h))
        parts.append(str(r))
        parts.append(str(t))
    return ' -> '.join(parts)


print('GreedyGNNRetriever defined.')

## 4. Approach A: RL Path Traversal

### Fixes vs previous version:
1. **Relation encoding**: action = (relation_emb + entity_emb), not just entity
2. **Shared budget across seeds**: single episode per question, multi-start
3. **Actual budget_ratio/step_ratio**: reflects real state

In [ ]:
class BudgetKGEnvironment:
    """MDP environment on local subgraph with shared budget across seeds."""
    
    STOP = '__STOP__'
    
    def __init__(self, gnn, max_steps=10, budget=50, top_k=20):
        self.gnn = gnn
        self.max_steps = max_steps
        self.budget = budget
        self.top_k = top_k
        # Episode state
        self.qid = None
        self.graph = None
        self.current_node = None
        self.visited = set()
        self.path = []
        self.triples = []
        self.step_count = 0
    
    def reset(self, qid):
        """Reset for new question. Does NOT start walking yet."""
        self.qid = qid
        self.graph = self.gnn.get_local_graph(qid)
        self.visited = set()
        self.path = []
        self.triples = []
        self.step_count = 0
        self.current_node = None
    
    def set_start(self, entity):
        """Set start node (called per seed, budget shared)."""
        self.current_node = entity
        self.visited.add(entity)
        self.path.append(entity)
    
    def get_actions(self):
        """Available actions: [(relation, neighbor, gnn_score), ..., STOP]."""
        if self.current_node is None or self.current_node not in self.graph:
            return [(self.STOP, None, None, 0.0)]
        
        neighbors = []
        for nb in self.graph.neighbors(self.current_node):
            if nb in self.visited:
                continue
            rel = self.graph[self.current_node][nb].get('relation', 'related_to')
            score = self.gnn.get_node_score(self.qid, nb)
            neighbors.append((rel, nb, rel, score))  # (relation, entity, relation_text, gnn_score)
        
        neighbors.sort(key=lambda x: -x[3])
        neighbors = neighbors[:self.top_k]
        # Format: (relation_text, entity, relation_text, gnn_score)
        actions = [(rel, nb, rel, score) for rel, nb, rel, score in neighbors]
        actions.append((self.STOP, None, None, 0.0))
        return actions
    
    def step(self, action_idx):
        actions = self.get_actions()
        action_idx = min(action_idx, len(actions) - 1)
        rel, next_node, _, gnn_score = actions[action_idx]
        
        if (rel == self.STOP or self.step_count >= self.max_steps
            or len(self.visited) >= self.budget):
            return True  # done
        
        self.triples.append((self.current_node, rel, next_node))
        self.current_node = next_node
        self.visited.add(next_node)
        self.path.append(next_node)
        self.step_count += 1
        
        done = (self.step_count >= self.max_steps
                or len(self.visited) >= self.budget
                or len(self.get_actions()) <= 1)
        return done
    
    @property
    def budget_ratio(self):
        return max(0, (self.budget - len(self.visited)) / self.budget)
    
    @property
    def step_ratio(self):
        return self.step_count / self.max_steps


print('BudgetKGEnvironment defined (shared budget).')

In [ ]:
class TraversalPolicy(nn.Module):
    """Policy for path traversal.
    
    FIX: action = concat(entity_emb, relation_emb, gnn_score)
    Previously: action = entity_emb only (relation ignored).
    """
    
    def __init__(self, emb_dim=384, hidden_dim=256, num_heads=4):
        super().__init__()
        self.emb_dim = emb_dim
        
        # State: query + node + history_attn + budget + step
        state_dim = emb_dim * 3 + 2
        self.state_enc = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Action: state_repr + entity_emb + relation_emb + gnn_score
        # FIX: +emb_dim for relation embedding
        action_dim = hidden_dim + emb_dim * 2 + 1
        self.action_scorer = nn.Sequential(
            nn.Linear(action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        
        self.history_attn = nn.MultiheadAttention(
            embed_dim=emb_dim, num_heads=num_heads, batch_first=True
        )
        
        self.stop_entity_emb = nn.Parameter(torch.randn(emb_dim))
        self.stop_relation_emb = nn.Parameter(torch.randn(emb_dim))
    
    def encode_history(self, query_emb, history_embs):
        if history_embs.shape[1] == 0:
            return torch.zeros(query_emb.shape[0], self.emb_dim, device=query_emb.device)
        q = query_emb.unsqueeze(1)
        out, _ = self.history_attn(q, history_embs, history_embs)
        return out.squeeze(1)
    
    def forward(self, query_emb, node_emb, history_embs,
                budget_ratio, step_ratio,
                action_entity_embs, action_relation_embs, gnn_scores):
        """Forward.
        
        Args:
            action_entity_embs: [batch, n_actions, emb_dim]
            action_relation_embs: [batch, n_actions, emb_dim]  <- NEW
            gnn_scores: [batch, n_actions, 1]
        """
        n_actions = action_entity_embs.shape[1]
        
        history_emb = self.encode_history(query_emb, history_embs)
        state = torch.cat([query_emb, node_emb, history_emb,
                          budget_ratio, step_ratio], dim=-1)
        state_repr = self.state_enc(state)
        
        state_exp = state_repr.unsqueeze(1).expand(-1, n_actions, -1)
        action_input = torch.cat([
            state_exp,
            action_entity_embs,
            action_relation_embs,  # NEW: relation embedding
            gnn_scores,
        ], dim=-1)
        
        scores = self.action_scorer(action_input).squeeze(-1)
        return F.softmax(scores, dim=-1)


policy_traversal = TraversalPolicy(emb_dim=EMB_DIM, hidden_dim=256)
print(f'TraversalPolicy params: {sum(p.numel() for p in policy_traversal.parameters()):,}')

In [ ]:
class TraversalRetriever:
    """RL traversal retriever with SHARED budget across seeds.
    
    FIX: single env.reset() per question, iterate seeds within same budget.
    FIX: action includes relation embedding.
    """
    
    def __init__(self, gnn, policy, embedder, max_steps=10, budget=50,
                 top_k=20, device='cpu'):
        self.gnn = gnn
        self.policy = policy.to(device)
        self.embedder = embedder
        self.max_steps = max_steps
        self.budget = budget
        self.top_k = top_k
        self.device = device
        self.env = BudgetKGEnvironment(gnn, max_steps, budget, top_k)
    
    def _encode_actions(self, actions):
        """Encode action list -> (entity_embs, relation_embs, gnn_scores)."""
        entity_embs = []
        relation_embs = []
        scores = []
        for rel, entity, rel_text, gnn_score in actions:
            if entity is None:  # STOP
                entity_embs.append(self.policy.stop_entity_emb.detach())
                relation_embs.append(self.policy.stop_relation_emb.detach())
            else:
                entity_embs.append(self.embedder.encode(entity))
                relation_embs.append(self.embedder.encode(rel_text))
            scores.append(gnn_score)
        
        return (
            torch.stack(entity_embs).unsqueeze(0).to(self.device),
            torch.stack(relation_embs).unsqueeze(0).to(self.device),
            torch.tensor(scores).unsqueeze(0).unsqueeze(-1).float().to(self.device),
        )
    
    def retrieve(self, qid, greedy=True):
        qdata = self.gnn.get(qid)
        if qdata is None:
            return {'triples': [], 'cost': 0, 'log_probs': [], 'context': ''}
        
        query_emb = self.embedder.encode(qdata['question']).to(self.device)
        seeds = self.gnn.get_seeds(qid)
        
        # SHARED budget: single reset, iterate seeds
        self.env.reset(qid)
        all_log_probs = []
        
        for seed in seeds:
            if seed in self.env.visited and seed != seeds[0]:
                continue  # Already visited from another seed
            if len(self.env.visited) >= self.budget:
                break
            
            self.env.set_start(seed)
            done = False
            
            while not done:
                actions = self.env.get_actions()
                if len(actions) <= 1:
                    break
                
                # Encode
                node_emb = self.embedder.encode(self.env.current_node).unsqueeze(0).to(self.device)
                history = self.embedder.encode_batch(self.env.path).unsqueeze(0).to(self.device)
                budget_r = torch.tensor([[self.env.budget_ratio]]).to(self.device)
                step_r = torch.tensor([[self.env.step_ratio]]).to(self.device)
                act_ent, act_rel, gnn_sc = self._encode_actions(actions)
                
                if greedy:
                    with torch.no_grad():
                        probs = self.policy(query_emb.unsqueeze(0), node_emb,
                                          history, budget_r, step_r,
                                          act_ent, act_rel, gnn_sc)
                    action_idx = probs.argmax(dim=-1).item()
                else:
                    probs = self.policy(query_emb.unsqueeze(0), node_emb,
                                      history, budget_r, step_r,
                                      act_ent, act_rel, gnn_sc)
                    dist = torch.distributions.Categorical(probs)
                    action_idx = dist.sample().item()
                    all_log_probs.append(dist.log_prob(
                        torch.tensor(action_idx, device=self.device)))
                
                done = self.env.step(action_idx)
        
        context = path_to_string(self.env.triples)
        return {
            'triples': self.env.triples,
            'cost': len(self.env.visited),
            'log_probs': all_log_probs,
            'context': context,
        }


print('TraversalRetriever defined (shared budget, relation encoding).')

## 5. Approach B: RL Path Selection

**Key insight**: GNN-RAG already generates candidate paths.
Instead of RL *traversal* on a small graph, use RL to *select* which paths to keep.

```
1. GNN scores entities -> top-K candidates
2. Shortest paths: seeds -> candidates (like GNN-RAG)
3. RL agent scores each path -> select top-B paths (budget B)
4. Selected paths -> verbalize -> LLM
```

Advantages over traversal:
- Builds directly on GNN-RAG (proven strong)
- No "small graph" problem
- Budget = number of paths (cleaner than node count)
- Easier optimization (score paths, not sequential walk)

In [ ]:
def extract_candidate_paths(qid, gnn, max_candidates=10, max_paths_per_pair=3):
    """Extract all candidate paths for a question (like GNN-RAG).
    
    Returns: list of {'path_triples': [...], 'candidate': str, 'gnn_score': float}
    """
    qdata = gnn.get(qid)
    if qdata is None:
        return []
    
    G = gnn.get_local_graph(qid)
    seeds = gnn.get_seeds(qid)
    candidates = qdata.get('top_candidates', [])[:max_candidates]
    
    all_paths = []
    seen = set()  # Deduplicate
    
    for seed in seeds:
        if seed not in G:
            continue
        for cand in candidates:
            if cand not in G or cand == seed:
                continue
            try:
                paths_gen = nx.all_shortest_paths(G, seed, cand)
                count = 0
                for p in paths_gen:
                    if count >= max_paths_per_pair:
                        break
                    # Convert to triples
                    triples = []
                    for i in range(len(p) - 1):
                        u, v = p[i], p[i+1]
                        rel = G[u][v].get('relation', 'related_to')
                        triples.append((u, rel, v))
                    
                    path_key = tuple((h, r, t) for h, r, t in triples)
                    if path_key not in seen:
                        seen.add(path_key)
                        all_paths.append({
                            'path_triples': triples,
                            'path_text': path_to_string(triples),
                            'candidate': cand,
                            'gnn_score': gnn.get_node_score(qid, cand),
                            'length': len(triples),
                        })
                    count += 1
            except nx.NetworkXNoPath:
                continue
    
    return all_paths


print('extract_candidate_paths defined.')

In [ ]:
class PathSelectionPolicy(nn.Module):
    """Score each candidate path for selection.
    
    Input per path: (query_emb, path_emb, gnn_score, path_length)
    Output: scalar score (higher = more useful)
    """
    
    def __init__(self, emb_dim=384, hidden_dim=256):
        super().__init__()
        self.emb_dim = emb_dim
        # Input: query_emb + path_emb + gnn_score + path_length
        self.scorer = nn.Sequential(
            nn.Linear(emb_dim * 2 + 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, query_emb, path_embs, gnn_scores, path_lengths):
        """Score paths.
        
        Args:
            query_emb: [batch, emb_dim]
            path_embs: [batch, n_paths, emb_dim]
            gnn_scores: [batch, n_paths, 1]
            path_lengths: [batch, n_paths, 1]
        
        Returns:
            selection_probs: [batch, n_paths] (sigmoid, independent per path)
        """
        n_paths = path_embs.shape[1]
        query_exp = query_emb.unsqueeze(1).expand(-1, n_paths, -1)
        
        x = torch.cat([query_exp, path_embs, gnn_scores, path_lengths], dim=-1)
        scores = self.scorer(x).squeeze(-1)
        return torch.sigmoid(scores)  # Independent selection per path


policy_selection = PathSelectionPolicy(emb_dim=EMB_DIM)
print(f'PathSelectionPolicy params: {sum(p.numel() for p in policy_selection.parameters()):,}')

In [ ]:
class PathSelectionRetriever:
    """RL-based path selection on top of GNN-RAG candidate paths.
    
    1. Extract candidate paths (GNN-RAG style)
    2. Score each path with PathSelectionPolicy
    3. Select top-B paths by score (budget B)
    4. Verbalize selected paths
    """
    
    def __init__(self, gnn, policy, embedder, max_paths=10, device='cpu'):
        self.gnn = gnn
        self.policy = policy.to(device)
        self.embedder = embedder
        self.max_paths = max_paths
        self.device = device
    
    def retrieve(self, qid, greedy=True):
        qdata = self.gnn.get(qid)
        if qdata is None:
            return {'triples': [], 'cost': 0, 'log_probs': [], 'context': '',
                    'selected_paths': [], 'all_paths': []}
        
        # Step 1: Extract candidate paths (GNN-RAG style)
        all_paths = extract_candidate_paths(qid, self.gnn)
        if not all_paths:
            return {'triples': [], 'cost': 0, 'log_probs': [], 'context': '',
                    'selected_paths': [], 'all_paths': all_paths}
        
        # Step 2: Encode paths
        query_emb = self.embedder.encode(qdata['question']).unsqueeze(0).to(self.device)
        
        path_embs = []
        gnn_scores = []
        path_lengths = []
        for p in all_paths:
            # Encode path as average of entity embeddings in path
            entities_in_path = set()
            for h, r, t in p['path_triples']:
                entities_in_path.add(h)
                entities_in_path.add(t)
            embs = self.embedder.encode_batch(list(entities_in_path))
            path_embs.append(embs.mean(dim=0))
            gnn_scores.append(p['gnn_score'])
            path_lengths.append(p['length'] / 5.0)  # Normalize
        
        path_embs_t = torch.stack(path_embs).unsqueeze(0).to(self.device)
        gnn_scores_t = torch.tensor(gnn_scores).unsqueeze(0).unsqueeze(-1).float().to(self.device)
        path_lengths_t = torch.tensor(path_lengths).unsqueeze(0).unsqueeze(-1).float().to(self.device)
        
        # Step 3: Score paths
        if greedy:
            with torch.no_grad():
                selection_probs = self.policy(query_emb, path_embs_t,
                                            gnn_scores_t, path_lengths_t)
        else:
            selection_probs = self.policy(query_emb, path_embs_t,
                                        gnn_scores_t, path_lengths_t)
        
        probs = selection_probs.squeeze(0)  # [n_paths]
        
        # Step 4: Select top-B paths
        log_probs_list = []
        if greedy:
            _, top_indices = probs.topk(min(self.max_paths, len(all_paths)))
            selected_indices = top_indices.tolist()
        else:
            # Sample from Bernoulli per path, keep up to max_paths
            selections = torch.bernoulli(probs)
            selected_indices = selections.nonzero(as_tuple=True)[0].tolist()
            if len(selected_indices) > self.max_paths:
                selected_indices = selected_indices[:self.max_paths]
            if not selected_indices:  # At least select top-1
                selected_indices = [probs.argmax().item()]
            
            # Log probs for REINFORCE
            for i in range(len(all_paths)):
                if i in selected_indices:
                    log_probs_list.append(torch.log(probs[i] + 1e-8))
                else:
                    log_probs_list.append(torch.log(1 - probs[i] + 1e-8))
        
        # Collect selected paths
        selected_paths = [all_paths[i] for i in selected_indices]
        all_triples = []
        for p in selected_paths:
            all_triples.extend(p['path_triples'])
        
        # Verbalize
        context_lines = [p['path_text'] for p in selected_paths]
        context = '\n'.join(context_lines)
        
        return {
            'triples': all_triples,
            'cost': len(selected_paths),  # Cost = number of paths
            'log_probs': log_probs_list,
            'context': context,
            'selected_paths': selected_paths,
            'all_paths': all_paths,
        }


print('PathSelectionRetriever defined.')

## 6. Reward Function (shared by both approaches)

In [ ]:
def compute_reward(collected_triples, gold_answers,
                   budget_used, total_budget,
                   alpha=0.6, beta=0.2, gamma=0.2,
                   question_emb=None, embedder=None):
    """Compute reward for RL episode.
    
    R = alpha * answer_quality + beta * efficiency + gamma * relevance
    """
    if not collected_triples:
        return -0.1
    
    # 1. Answer quality (entity overlap)
    collected_entities = set()
    for h, r, t in collected_triples:
        collected_entities.add(str(h))
        collected_entities.add(str(t))
    
    gold_set = set()
    for a in gold_answers:
        gold_set.add(a.get('text', a.get('kb_id', str(a))) if isinstance(a, dict) else str(a))
    
    overlap = len(collected_entities & gold_set)
    answer_quality = min(overlap / max(len(gold_set), 1), 1.0)
    
    # 2. Efficiency
    efficiency = 1.0 - (budget_used / total_budget)
    
    # 3. Relevance (optional, skip if no embedder for speed)
    relevance = 0.0
    if question_emb is not None and embedder is not None:
        ctx = path_to_string(collected_triples)
        ctx_emb = embedder.model.encode(ctx, convert_to_tensor=True)
        q_emb = question_emb if isinstance(question_emb, torch.Tensor) else torch.tensor(question_emb)
        relevance = max(0, F.cosine_similarity(ctx_emb.unsqueeze(0), q_emb.unsqueeze(0)).item())
    
    return alpha * answer_quality + beta * efficiency + gamma * relevance


print('Reward function defined.')

## 7. Test All Retrievers

In [ ]:
# Load GNN features (use TEST for demo, TRAIN for actual training)
gnn_test = GNNHeuristic(os.path.join(CHECKPOINT_DIR, 'gnn_features_test.pkl'))

if gnn_test.features:
    qid = list(gnn_test.features.keys())[0]
    qdata = gnn_test.get(qid)
    print(f'Question: {qdata["question"]}')
    print(f'Seeds: {gnn_test.get_seeds(qid)}')
    print(f'Answers: {gnn_test.get_answers(qid)}')
    
    # --- Greedy GNN baseline ---
    greedy = GreedyGNNRetriever(gnn_test, max_steps=5, budget=20)
    r1 = greedy.retrieve(qid)
    print(f'\n--- GreedyGNN: {len(r1["triples"])} triples, cost={r1["cost"]} ---')
    
    # --- Traversal RL (random policy) ---
    traversal = TraversalRetriever(gnn_test, policy_traversal, embedder,
                                   max_steps=5, budget=20)
    r2 = traversal.retrieve(qid, greedy=True)
    print(f'\n--- Traversal RL: {len(r2["triples"])} triples, cost={r2["cost"]} ---')
    
    # --- Path Selection RL ---
    selector = PathSelectionRetriever(gnn_test, policy_selection, embedder,
                                      max_paths=5)
    r3 = selector.retrieve(qid, greedy=True)
    print(f'\n--- PathSelection RL: {len(r3["triples"])} triples, '
          f'{len(r3["selected_paths"])}/{len(r3["all_paths"])} paths selected ---')
    print(f'Context:\n{r3["context"][:300]}')
else:
    print('No GNN features. Run P2 first.')

## Summary P4a

| Component | What | Key Fix |
|-----------|------|--------|
| `GreedyGNNRetriever` | **NEW baseline** | Proves whether RL adds value |
| `TraversalPolicy` | RL traversal policy | **FIX: relation embedding in actions** |
| `TraversalRetriever` | RL traversal | **FIX: shared budget across seeds** |
| `PathSelectionPolicy` | **NEW: path scoring** | Alternative to traversal |
| `PathSelectionRetriever` | **NEW: select paths** | Builds on GNN-RAG output |
| `compute_reward()` | Reward function | Cleaner, entity overlap proxy |

### P4b will compare:
1. GreedyGNN (no RL)
2. GRL-RAG Traversal (RL walks on graph)
3. GRL-RAG Selection (RL selects paths)

If GreedyGNN ~= Traversal RL → RL traversal not needed.
If Selection > Traversal → use Selection as main method.

In [ ]:
# === Save grl_rag.py to Drive utils (cho P4b import) ===
# File grl_rag.py chua tat ca classes: GNNHeuristic, Embedder, policies, retrievers, reward
# P4b chi can: from grl_rag import *

import shutil
GRL_RAG_SRC = '/content/drive/MyDrive/ThS/Research/colab/graph_rag/grl_rag.py'

# Copy tu local repo (neu chay local) hoac write truc tiep
# File nay duoc sync tu colab/graph_rag/grl_rag.py
if not os.path.exists(GRL_RAG_SRC):
    # Download from repo or copy
    print(f'grl_rag.py not found at {GRL_RAG_SRC}')
    print('Upload file grl_rag.py vao folder utils/ tren Drive')
else:
    print(f'grl_rag.py OK: {GRL_RAG_SRC}')

# Verify import
try:
    from grl_rag import GNNHeuristic, Embedder, TraversalPolicy, PathSelectionPolicy
    print('Import grl_rag: OK')
except ImportError:
    print('grl_rag chua co trong sys.path - upload file truoc')